In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Set the folder where the split CSV files are saved.
print("\nFile Loading...\n")
split_folder = Path(r"data\split data")

# Check training files from newest to oldest.
train_files = sorted(
    split_folder.glob("latest_data_train_*.csv"),
    reverse=True,
)

for train_path in train_files:
    timestamp = train_path.stem.removeprefix("latest_data_train_")
    test_path = split_folder / f"latest_data_test_{timestamp}.csv"

    if test_path.is_file():
        break
else:
    raise FileNotFoundError(
        f"No matching training and test CSV files found in {split_folder}"
    )

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

total_samples = len(train) + len(test)
actual_train_pct = 100 * len(train) / total_samples

train_pct = int(round(actual_train_pct, -1))
test_pct = 100 - train_pct

print(f"Training file: {train_path.name}")
print(f"Training dataset: {train.shape[0]} samples, {train.shape[1]} parameters")
print(f"Test file: {test_path.name}")
print(f"Test dataset: {test.shape[0]} samples, {test.shape[1]} parameters")
print(f"Estimate split ratio (train:test): \033[31m{train_pct}:{test_pct}\033[0m")
print("\033[32mDone! \033[0m")

In [ ]:
#Define input and output target
TARGET = "moisture_chl"

X_train = train.drop(columns=[TARGET, 'tlabel'])
y_train = train[TARGET]

X_test = test.drop(columns=[TARGET, 'tlabel'])
y_test = test[TARGET]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# Scale the selected features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.neural_network import MLPRegressor # Load package for MLP
from sklearn.ensemble import (
    GradientBoostingRegressor, # Load package for GBR
    RandomForestRegressor # Load package for RF
)

from sklearn.neighbors import KNeighborsRegressor # Load package for KNN
from sklearn.linear_model import ElasticNet # Load package for LR(Elastic Net)
from sklearn.svm import SVR # Load package for SVR
from xgboost import XGBRegressor # Load package for XGBOOST

# define models
models = {

    # --------------------------------------------------------
    # 1. MLP
    # --------------------------------------------------------
    'MLP': MLPRegressor(
        hidden_layer_sizes=(32, 16),
        alpha=1.0,
        activation='relu',
        random_state=42,
        solver='lbfgs',
        max_iter=800
    ),

    # --------------------------------------------------------
    # 2. Gradient Boosting
    # --------------------------------------------------------
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        subsample=0.7,
        random_state=42
    ),

    # --------------------------------------------------------
    # 3. KNN
    # --------------------------------------------------------
    'KNN': KNeighborsRegressor(
        n_neighbors=3,
        weights='uniform',
        metric='manhattan'
    ),

    # --------------------------------------------------------
    # 4. ElasticNet
    # --------------------------------------------------------
    'ElasticNet': ElasticNet(
        alpha=0.1,
        l1_ratio=0.7,
        random_state=42,
        max_iter=20000
    ),

    # --------------------------------------------------------
    # 5. Random Forest
    # --------------------------------------------------------
    'RandomForest': RandomForestRegressor(
        n_estimators=50,
        max_depth=6,
        max_features=1.0,
        min_samples_leaf=3,
        random_state=42
    ),

    # --------------------------------------------------------
    # 6. SVR
    # --------------------------------------------------------
    'SVR': SVR(
        kernel='linear',
        C=0.03,
        epsilon=0.5
    ),

    # --------------------------------------------------------
    # 7. XGBoost
    # --------------------------------------------------------
    'XGBoost': XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    )
}

In [14]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone

# Match actual CHL values and transformer labels to the test feature rows.
# This assumes X_test and test share the original DataFrame index.
test_index = X_test.index
tx_labels = test.loc[test_index, "tlabel"].to_numpy()

if isinstance(y_test, pd.Series):
    actual = y_test.loc[test_index].to_numpy(dtype=float)
else:
    # If y_test is an array, it must already follow the row order of X_test.
    actual = np.asarray(y_test, dtype=float).reshape(-1)

if len(actual) != len(X_test):
    raise ValueError("y_test and X_test must contain the same number of rows.")

# Percentage error is undefined when the actual CHL is zero.
nonzero_actual = actual != 0

trained_models = {}
metric_rows = []
prediction_tables = []
parameter_rows = []

for model_name, estimator in models.items():
    model = clone(estimator)
    model.fit(X_train_scaled, y_train)
    predicted = model.predict(X_test_scaled)
    trained_models[model_name] = model

    for parameter, value in model.get_params(deep=False).items():
        parameter_rows.append({
            "Model": model_name,
            "Parameter": parameter,
            "Value": repr(value),
        })

    absolute_error = np.abs(predicted - actual)
    errors = np.divide(
        100 * absolute_error,
        np.abs(actual),
        out=np.full(actual.shape, np.nan),
        where=nonzero_actual,
    )

    metric_rows.append({
        "Model": model_name,
        "R²": r2_score(actual, predicted) if len(actual) >= 2 else np.nan,
        "RMSE (percentage points)": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "MAE (percentage points)": mean_absolute_error(actual, predicted),
        "MAPE (%)": errors[nonzero_actual].mean() if nonzero_actual.any() else np.nan,
    })

    prediction_tables.append(pd.DataFrame({
        "Model": model_name,
        "TX Label": tx_labels,
        "Test CHL (%)": actual,
        "Predicted CHL (%)": predicted,
        "Absolute Error (percentage points)": absolute_error,
        "Error (%)": errors,
    }))

    print(f"{model_name} training: \033[32mDone!\033[0m")

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_tables, ignore_index=True)

print("\nModel Performance")
display(metrics.style.hide(axis="index").format(precision=4, na_rep="—"))

print("\nActual vs Predicted CHL")
display(predictions.style.hide(axis="index").format(precision=4, na_rep="—"))

MLP training: Done!
GradientBoosting training: Done!
KNN training: Done!
ElasticNet training: Done!
RandomForest training: Done!
SVR training: Done!
XGBoost training: Done!

Model Performance


Model,R²,RMSE (percentage points),MAE (percentage points),MAPE (%)
MLP,0.1708,1.0073,0.7970,50.4278
GradientBoosting,0.3710,0.8773,0.6620,44.1925
KNN,0.6319,0.6711,0.5694,29.0886
ElasticNet,0.4368,0.8302,0.7296,44.2212
RandomForest,0.4974,0.7842,0.6131,42.8662
SVR,0.4859,0.7931,0.6857,38.0323
XGBoost,0.5117,0.7730,0.6750,34.7173



Actual vs Predicted CHL


Model,TX Label,Test CHL (%),Predicted CHL (%),Absolute Error (percentage points),Error (%)
MLP,T1,3.4500,3.6217,0.1717,4.9764
MLP,T6,3.3000,3.8108,0.5108,15.4778
MLP,T12,4.1000,3.0810,1.0190,24.8531
MLP,T2,0.9000,2.8090,1.9090,212.1124
MLP,T9,1.6000,1.7231,0.1231,7.6960
MLP,T17,2.8000,1.7514,1.0486,37.4512
GradientBoosting,T1,3.4500,3.4712,0.0212,0.6150
GradientBoosting,T6,3.3000,3.7804,0.4804,14.5585
GradientBoosting,T12,4.1000,2.9423,1.1577,28.2377
GradientBoosting,T2,0.9000,2.5617,1.6617,184.6301


In [11]:
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
TOP_N = 10
importance_tables = []

# `actual` is the CHL target aligned to X_test in the previous cell.
for model_name, fitted_model in trained_models.items():
    result = permutation_importance(
        fitted_model,
        X_test_scaled,
        actual,
        n_repeats=30,
        random_state=RANDOM_STATE,
        scoring="neg_mean_absolute_error",
        n_jobs=1,
    )

    mean_mae_change = result.importances_mean
    positive_increases = np.clip(mean_mae_change, 0, None)
    total_increase = positive_increases.sum()

    importance_percent = np.zeros_like(positive_increases)
    if total_increase > 0:
        importance_percent = 100 * positive_increases / total_increase

    model_importance = (
        pd.DataFrame({
            "Model": model_name,
            "Feature": X_train.columns,
            "Mean MAE change (percentage points)": mean_mae_change,
            "SD across shuffles (percentage points)": result.importances_std,
            "Importance (%)": importance_percent,
        })
        .sort_values("Mean MAE change (percentage points)", ascending=False)
    )

    importance_tables.append(model_importance)

feature_importance = pd.concat(importance_tables, ignore_index=True)
top_features = feature_importance.groupby("Model", sort=False).head(TOP_N)

print(f"\nTop {TOP_N} features for each model")
display(
    top_features.style.hide(axis="index").format({
        "Mean MAE change (percentage points)": "{:+.4f}",
        "SD across shuffles (percentage points)": "{:.4f}",
        "Importance (%)": "{:.2f}%",
    })
)


Top 10 features for each model


Model,Feature,Mean MAE change (percentage points),SD across shuffles (percentage points),Importance (%)
MLP,BV_moisture,+0.1274,0.0532,17.07%
MLP,BV_percent_relative_saturation,+0.1158,0.0812,15.51%
MLP,appearance_label,+0.1034,0.1731,13.85%
MLP,BV_N2,+0.0714,0.0627,9.56%
MLP,BV_bdv,+0.0551,0.0964,7.37%
MLP,BV_color,+0.0547,0.0249,7.32%
MLP,BV_PF_oxidation,+0.0411,0.0321,5.51%
MLP,BV_Oil_Moisture,+0.0370,0.0523,4.95%
MLP,BV_PF,+0.0343,0.0178,4.60%
MLP,BV_resistivity,+0.0259,0.0160,3.46%


In [12]:
from datetime import datetime
from pathlib import Path

import joblib

saved_at = datetime.now().astimezone()
timestamp = saved_at.strftime("%Y%m%d_%H%M%S_%f")

model_folder = (Path("models") / timestamp).resolve()
model_folder.mkdir(parents=True, exist_ok=False)

input_features = X_train.columns.tolist()
saved_model_paths = {}

print("\nSaving trained models...")

for model_name, fitted_model in trained_models.items():
    model_path = model_folder / f"{model_name.lower()}_chl_model.joblib"

    model_bundle = {
        "model_name": model_name,
        "model": fitted_model,
        "scaler": scaler,
        "input_features": input_features,
        "target": "moisture_chl",
        "saved_at": saved_at.isoformat(timespec="seconds"),
    }

    joblib.dump(model_bundle, model_path)
    saved_model_paths[model_name] = model_path

    print(f"{model_name}: \033[32mSaved!\033[0m")

print(f"\nSaved {len(saved_model_paths)} models to {model_folder}")


Saving trained models...
MLP: Saved!
GradientBoosting: Saved!
KNN: Saved!
ElasticNet: Saved!
RandomForest: Saved!
SVR: Saved!
XGBoost: Saved!

Saved 7 models to C:\Users\USER\Desktop\TRAINING\Different Model\models\20261114_161353_671670


In [13]:
from datetime import datetime
from pathlib import Path

metrics = (
    pd.DataFrame(metric_rows)
    .sort_values("MAE (percentage points)", kind="stable")
    .reset_index(drop=True)
)

predictions = pd.concat(prediction_tables, ignore_index=True)

model_parameters = (
    pd.DataFrame(parameter_rows)
    .sort_values(["Model", "Parameter"])
    .reset_index(drop=True)
)

importance_export = feature_importance
if importance_export.empty:
    importance_export = pd.DataFrame({
        "Status": ["Permutation importance was unavailable for this run."]
    })

result_folder = Path("result")
result_folder.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
result_path = result_folder / f"CHL_Model_Results_{timestamp}.xlsx"

tables = {
    "Model Performance": metrics,
    "Actual vs Predicted": predictions,
    "Feature Importance": importance_export,
    "Model Parameters": model_parameters,
}

with pd.ExcelWriter(result_path, engine="openpyxl") as writer:
    for sheet_name, table in tables.items():
        table.to_excel(writer, sheet_name=sheet_name, index=False)
        sheet = writer.sheets[sheet_name]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions

print(f"\n\033[32mResults saved to {result_path.resolve()}\033[0m")


Results saved to C:\Users\USER\Desktop\TRAINING\Different Model\result\CHL_Model_Results_20261114_161353_863296.xlsx
